In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import os

In [2]:
df = pd.read_csv('../../GWIorgs_v4.csv')

In [3]:
print(f"There are {len(df)} number of non-profit organizations serving to Lawrence residents")

print("-"*50)

for i in range(len(df)):
    print(f"{df['Name'][i]} - {df['URL'][i]}")

There are 65 number of non-profit organizations serving to Lawrence residents
--------------------------------------------------
ACT Lawrence - https://www.actlawrence.org
Bellesini Academy - https://bellesiniacademy.org
Beyond Soccer - https://www.beyondsoccerlawrence.org
Bread and Roses Housing - https://brhousing.org
Bread and Roses Kitchen - https://breadandroseskitchen.org
Children's Friend and Family Services- a division of JRI - https://www.childrensfriend.net
Community Giving Tree - https://www.communitygivingtree.org
Community InRoads - https://www.communityinroads.org
EforAll - https://eforall.org/ma/merrimack-valley
Elevated Thought - https://www.elevatedthought.org
Elliot Community Services - https://www.eliotchs.org/services
Esperanza Academy - https://esperanzaacademy.org
Essex County Habitat For Humanity - https://www.essexcountyhabitat.org/shop
Family Services of Merrimack Valley/LMCC - https://fsmv.org
Greater Lawrence Community Action Council (GLCAC) - https://www.glc

In [4]:
#first website as a test

url = df.loc[0, 'URL']
page = requests.get(url)
print(page.status_code)

soup = BeautifulSoup(page.text, 'html.parser')
#print(soup.get_text())

#get the hyperlinks
links = soup.find_all('a')
print(len(set(links)))
for link in links[:10]:
    print(link.get("href"))


200
56
https://www.actlawrence.org
https://www.actlawrence.org
https://www.actlawrence.org
https://www.actlawrence.org/about-5
https://www.actlawrence.org/team
https://www.actlawrence.org/about-1
https://www.actlawrence.org/sponsors
https://www.actlawrence.org/housing
https://www.actlawrence.org/fthb
https://www.actlawrence.org/homeowner-resources


In [5]:
KEYWORDS = ['about', 'team', 'resource', 'housing', 'service', 'education', 'philosophy', 'address', 'contact', 'who', 'support']
good_links =[]

for link in links:
    href = link.get('href')
    if href:
        for keyword in KEYWORDS:
            if keyword in href:
                good_links.append(href)
                break
            
print(good_links)
print(len(good_links))

['https://www.actlawrence.org/about-5', 'https://www.actlawrence.org/team', 'https://www.actlawrence.org/about-1', 'https://www.actlawrence.org/housing', 'https://www.actlawrence.org/homeowner-resources', 'https://www.actlawrence.org/financial-literacy-and-rental-support', 'https://www.actlawrence.org/copy-of-rental-resources-and-assistan', 'https://www.actlawrence.org/housing', 'https://www.actlawrence.org/event-details/landlord-education-workshop-in-person-6', 'https://www.actlawrence.org/event-details/landlord-education-workshop-in-person-6', 'https://www.actlawrence.org/event-details/financial-education-workshop-educacion-financiera-3', 'https://www.actlawrence.org/event-details/financial-education-workshop-educacion-financiera-3', 'https://www.actlawrence.org/event-details/financial-education-workshop-educacion-financiera-4', 'https://www.actlawrence.org/event-details/financial-education-workshop-educacion-financiera-4', 'https://www.actlawrence.org/event-details/landlord-educatio

In [6]:
all_text = ""

for link in list(set(good_links)):
    page = requests.get(link)
    soup = BeautifulSoup(page.text, 'html.parser')
    text = soup.get_text(" ", strip = True)
    all_text += text

print(len(all_text))
    

19042


In [18]:
all_text_lower = all_text.lower()
listed = df.loc[0, "Services"].lower()



SERVICE_KEYWORDS = {
    "Food": ["food pantry", "meals", "soup kitchen", "hunger"],
    "Housing": ["housing", "shelter", "homeless", "homeowner"],
    "Health": ["mental health", "counseling", "clinic", "medical"],
    "Youth": ["youth", "after school", "summer camp"],
    "Jobs": ["workforce", "job training", "employment", "financial"],
    "Legal": ["legal", "immigration", "citizenship"],
}

missing = []

for category, services in SERVICE_KEYWORDS.items():
    for service in services:
        if service in all_text_lower and category.lower() not in listed:
            missing.append(category)
            break
        
print(f"Missing: {missing}")


Missing: ['Housing', 'Health', 'Youth', 'Jobs', 'Legal']
